# Oppitunti 01 - Johdatus tekoälyagetteihin

Tervetuloa ensimmäiseen oppituntiin **AI Agents for Beginners** -kurssilla!

**Tekoälyagentti** on ohjelma, joka käyttää suurta kielimallia (LLM) päättelymoottorinaan ja voi tehdä *toimia* todellisessa maailmassa — soittaa rajapintoja, tehdä tietokantahakuja tai suorittaa koodia — saavuttaakseen tavoitteen käyttäjän puolesta.

Tässä muistikirjassa rakennat ensimmäisen agenttisi: **Matka-agentin**, joka suosittelee lomakohteita. Matkan varrella opit:

1. Yhdistämään Microsoft Foundry Agent Serviceen käyttäen **Microsoft Agent Frameworkia**.
2. Antamaan agentille **työkalun** — tavallisen Python-funktion, jota se voi kutsua.
3. Käynnistämään agentin ja tarkastelemaan sen vastausta.
4. Striimaamaan agentin vastaus token tokenilta.


## Asetukset

Ennen tämän muistiokirjan suorittamista varmista, että sinulla on:

1. **Microsoft Foundry -projekti**, jossa on otettu käyttöön chat-malli (esim. `gpt-4.1-mini`).
2. **Olet kirjautunut sisään Azure CLI:llä** — suorita `az login` päätelaitteessasi.
3. **Aseta tarvittavat ympäristömuuttujat:**
   - `AZURE_AI_PROJECT_ENDPOINT` — Microsoft Foundry -projektisi päätepiste.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — käyttöönotetun mallisi nimi.

Alla oleva solu asentaa tarvitsemasi Python-kirjastot.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from agent_framework import tool

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Ensimmäisen agenttisi luominen

Agentti tarvitsee kaksi asiaa:

- **Ohjeet**, jotka kertovat *kuka se on* ja *miten sen tulee käyttäytyä* (järjestelmäkehotus).
- **Työkalut** — Python-funktiot, jotka on koristeltu `@tool`-koristeella, ja joita agentti voi kutsua saadakseen tietoa tai suorittaakseen toimintoja.

Alla määrittelemme yksinkertaisen työkalun, joka palauttaa listan suosituista lomakohteista. Agentti käyttää tätä työkalua, kun käyttäjä pyytää matkasuosituksia.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = provider.as_agent(
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
    tools=[get_destinations],
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Virtaavat vastaukset

Vuorovaikutteisemman kokemuksen saamiseksi voit **striimata** agentin vastauksen. Sen sijaan, että odottaisit koko vastausta, agentti tuottaa tekstin osia sitä mukaa kun ne luodaan. Tämä on erityisen hyödyllistä chat-käyttöliittymissä, joissa haluat näyttää tuloksen reaaliajassa.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Yhteenveto

Tässä oppitunnissa opit, kuinka:

- **Luo tarjoaja**, joka yhdistää Microsoft Foundry Agent Serviceen `FoundryChatClient`-luokan avulla.
- **Määrittele työkalu** `@tool`-koristelijan avulla, jotta agentti voi kutsua Python-funktioitasi.
- **Käynnistä agentti** käyttäjäviestillä ja tulosta sen vastaus.
- **Lähetä vastauksia striimaten** reaaliaikaista tulostusta varten.

Seuraavassa oppitunnissa tutkimme agenttirajapintoja tarkemmin ja opimme, kuinka antaa agenteille tehokkaampia työkaluja ja monivaiheisia päättelymahdollisuuksia.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Vaikka pyrimme tarkkuuteen, otathan huomioon, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäinen asiakirja sen alkuperäiskielellä on virallinen lähde. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
